## classification problem

In [2]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier,DecisionTreeRegressor
# from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.metrics import accuracy_score,log_loss,r2_score
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression,LinearRegression,ElasticNet
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.compose import ColumnTransformer
from sklearn.compose import make_column_selector
from tqdm import tqdm
from sklearn.neighbors import KNeighborsRegressor

In [3]:
cancer=pd.read_csv("D:\\AshleshaRuchika\\PGCP-AI\\Machine Learning\\Wisconsin\\BreastCancer.csv")
cancer

,Code,Clump,UniCell_Size,Uni_CellShape,MargAdh,SEpith,BareN,BChromatin,NoemN,Mitoses,Class
0,61634,5,4,3,1,2,2,2,3,1,Benign
1,63375,9,1,2,6,4,10,7,7,2,Malignant
2,76389,10,4,7,2,2,8,6,1,1,Malignant
3,95719,6,10,10,10,8,10,7,10,7,Malignant
4,128059,1,1,1,1,2,5,5,1,1,Benign
...,...,...,...,...,...,...,...,...,...,...,...
694,1369821,10,10,10,10,5,10,10,10,7,Malignant
695,1371026,5,10,10,10,4,10,5,6,3,Malignant
696,1371920,5,1,1,1,2,1,3,2,1,Benign
697,8233704,4,1,1,1,1,1,2,1,1,Benign


In [4]:
X,y=cancer.drop("Class",axis=1),cancer["Class"]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=26,stratify=cancer["Class"])

In [13]:
n_est=[10,25,50,75,100]
rate=np.linspace(0.0001,0.9,30)
depth=[2,3,4,5,6]
scores=[]
for n in tqdm(n_est):
    for r in rate:
        for d in depth:
            gdm=GradientBoostingClassifier(random_state=26,n_estimators=n,learning_rate=r,max_depth=d)
            gdm.fit(X_train,y_train)
            y_pred_prob=gdm.predict_proba(X_test)
            scores.append([n,r,d,log_loss(y_test,y_pred_prob)])
df_scores=pd.DataFrame(scores,columns=["n_est","rate","depth","score"])
df_scores.sort_values("score")            

100%|████████████████████████████████████████████████████████████████████████████████████| 5/5 [01:15<00:00, 15.18s/it]


,n_est,rate,depth,score
115,10,0.713814,2,0.090715
285,25,0.837938,2,0.094103
290,25,0.868969,2,0.096980
90,10,0.558659,2,0.098586
105,10,0.651752,2,0.099256
...,...,...,...,...
4,10,0.000100,6,0.642041
3,10,0.000100,5,0.642046
2,10,0.000100,4,0.642065
1,10,0.000100,3,0.642092


#  apply on HR dataset

In [14]:
hr=pd.read_csv("D:\\AshleshaRuchika\\PGCP-AI\\Machine Learning\\HR_comma_sep.csv")
hr

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,Department,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.10,0.77,6,247,4,0,1,0,sales,low
3,0.92,0.85,5,259,5,0,1,0,sales,low
4,0.89,1.00,5,224,5,0,1,0,sales,low
...,...,...,...,...,...,...,...,...,...,...
14990,0.40,0.57,2,151,3,0,1,0,support,low
14991,0.37,0.48,2,160,3,0,1,0,support,low
14992,0.37,0.53,2,143,3,0,1,0,support,low
14993,0.11,0.96,6,280,4,0,1,0,support,low


In [15]:
X,y=hr.drop("left",axis=1),hr["left"]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=26,stratify=y)

In [16]:
ohe=OneHotEncoder(sparse_output=False,drop="first").set_output(transform="pandas")
transf=ColumnTransformer(transformers=[("OHE",ohe, make_column_selector
                                        (dtype_include=object))],remainder="passthrough",verbose_feature_names_out=False).set_output(transform="pandas")
X_trn_ohe=transf.fit_transform(X_train)
X_tst_ohe=transf.transform(X_test)

In [17]:
n_est=[10,25,50,75,100]
rate=np.linspace(0.0001,0.9,30)
depth=[2,3,4,5,6]
scores=[]
for n in tqdm(n_est):
    for r in rate:
        for d in depth:
            gdm=GradientBoostingClassifier(random_state=26,n_estimators=n,learning_rate=r,max_depth=d)
            gdm.fit(X_trn_ohe,y_train)
            y_pred_prob=gdm.predict_proba(X_tst_ohe)
            scores.append([n,r,d,log_loss(y_test,y_pred_prob)])
df_scores=pd.DataFrame(scores,columns=["n_est","rate","depth","score"])
df_scores.sort_values("score")  

100%|███████████████████████████████████████████████████████████████████████████████████| 5/5 [10:32<00:00, 126.53s/it]


,n_est,rate,depth,score
624,100,0.124224,6,0.055132
629,100,0.155255,6,0.055249
489,75,0.217317,6,0.055700
339,50,0.217317,6,0.056577
334,50,0.186286,6,0.056745
...,...,...,...,...
578,75,0.775876,5,0.841349
447,50,0.900000,4,2.660387
597,75,0.900000,4,2.695487
747,100,0.900000,4,2.830960


In [ ]:
n_est=[10,25,50,75,100]
rate=np.linspace(0.0001,0.9,30)
depth=[2,3,4,5,6]
scores=[]
for n in tqdm(n_est):
    for r in rate:
        for d in depth:
            gdm=LGBMClassifier(random_state=26,n_estimators=n,learning_rate=r,max_depth=d,verbose=-1)
            gdm.fit(X_trn_ohe,y_train)
            y_pred_prob=gdm.predict_proba(X_tst_ohe)
            scores.append([n,r,d,log_loss(y_test,y_pred_prob)])
df_scores=pd.DataFrame(scores,columns=["n_est","rate","depth","score"])
df_scores.sort_values("score") 